In [10]:
import os
import ipywidgets as widgets
import pandas as pd
from IPython.display import clear_output, display
df = None

txt_path = widgets.Text(
    value="",
    placeholder=r"Наприклад: C:\Users\User\Documents\data.csv",
    description="Шлях до CSV:",
    style={"description_width": "initial"},
)

txt_path.layout = widgets.Layout(
    width="65%", min_width="300px", margin="0px 15px 0px 0px"
)

btn_load = widgets.Button(
    description="Завантажити дані", button_style="primary", icon="download"
)
btn_load.layout = widgets.Layout(width="180px", height="32px")

lbl_status = widgets.Label(
    value="Будь ласка, введіть шлях до CSV-файлу на ПК та натисніть кнопку."
)
lbl_status.layout = widgets.Layout(margin="12px 0px 5px 5px")

output_dataframe = widgets.Output()
output_dataframe.layout = widgets.Layout(
    margin="15px 0px 0px 0px", max_height="250px"
)

def load_csv_data(b):
    global df
    with output_dataframe:
        clear_output()

    raw_path = txt_path.value.strip().strip('"').strip("'")

    if not raw_path:
        lbl_status.value = "Помилка: Поле вводу шляху порожнє!"
        return

    normalized_path = os.path.normpath(raw_path)

    if not os.path.exists(normalized_path):
        lbl_status.value = (
            f"Помилка: Файл не знайдено за адресою: {raw_path}"
        )
        return

    if not os.path.isfile(normalized_path):
        lbl_status.value = "Помилка: Вказаний шлях веде до папки."
        return

    try:
        temp_df = pd.read_csv(normalized_path, encoding="utf-8")

        if temp_df.empty:
            lbl_status.value = "Помилка: Файл порожній або не містить даних."
            return

        df = temp_df
        lbl_status.value = f"Дані успішно завантажено! Розмірність таблиці: {df.shape[0]} рядків на {df.shape[1]} стовпців."

        with output_dataframe:
            print("Перші 5 рядків отриманих даних (df.head):")
            display(df.head(5))

    except pd.errors.EmptyDataError:
        lbl_status.value = "Помилка: Файл порожній або містить некоректні дані."
    except pd.errors.ParserError:
        lbl_status.value = (
            "Помилка: Файл неможливо прочитати. Перевірте розділювачі (має бути кома)."
        )
    except Exception as e:
        lbl_status.value = f"Помилка читання файлу: {str(e)}"

btn_load.on_click(load_csv_data)

form_row = widgets.HBox([txt_path, btn_load])
form_row.layout.align_items = "center"

load_interface_vbox = widgets.VBox([form_row, lbl_status, output_dataframe])
load_interface_vbox.layout = widgets.Layout(
    padding="15px",
    border="1px solid #cfd8dc",
    border_radius="6px",
    background_color="#f8fafc",
    margin="0px 0px 20px 0px",
)
display(load_interface_vbox)

In [11]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from IPython.display import clear_output, display
from ipywidgets import AppLayout
from scipy.interpolate import make_interp_spline

if df is None:
    print(
        "Помилка: DataFrame порожній! Спочатку завантажте дані у Першій клітинці."
    )
else:
    sns.set_theme(style="whitegrid")
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()

    if not numeric_columns:
        print("Помилка: У DataFrame відсутні числове стовпці для аналізу.")
    else:
        max_index = len(df) - 1

        dropdown_column = widgets.Dropdown(
            options=numeric_columns, description="Параметр:"
        )
        dropdown_column.layout.width = "100%"

        slider_width = widgets.IntSlider(
            value=2, min=1, max=5, description="Товщина лінії:"
        )
        slider_width.layout.width = "100%"

        picker_color = widgets.ColorPicker(
            concise=False, description="Колір лінії:", value="#1e3a8a"
        )
        picker_color.layout.width = "100%"

        slider_range = widgets.IntRangeSlider(
            value=[0, min(100, max_index)], min=0, max=max_index, description="Діапазон:"
        )
        slider_range.layout.width = "100%"

        left_sidebar_panel = widgets.VBox(
            [
                widgets.HTML("<b>Параметри графіка</b>"),
                dropdown_column,
                picker_color,
                slider_width,
                slider_range,
            ],
            layout=widgets.Layout(
                padding="10px",
                border="1px solid #cbd5e1",
                border_radius="6px",
                background_color="#f8fafc",
            ),
        )
        lbl_count = widgets.Label(value="-")
        lbl_min = widgets.Label(value="-")
        lbl_max = widgets.Label(value="-")
        lbl_mean = widgets.Label(value="-")
        lbl_median = widgets.Label(value="-")
        stats_grid = widgets.GridBox(
            [
                widgets.HTML("<b>Кількість:</b>"), lbl_count,
                widgets.HTML("<b>Мінімум:</b>"), lbl_min,
                widgets.HTML("<b>Максимум:</b>"), lbl_max,
                widgets.HTML("<b>Середнє:</b>"), lbl_mean,
                widgets.HTML("<b>Медіана:</b>"), lbl_median,
            ],
            layout=widgets.Layout(
                grid_template_columns="90px 1fr",
                grid_row_gap="8px",
                padding="5px",
                width="100%",
            ),
        )

        right_sidebar_panel = widgets.VBox(
            [
                widgets.HTML("<b>Характеристики</b>"),
                stats_grid,
            ],
            layout=widgets.Layout(
                padding="10px",
                border="1px solid #cbd5e1",
                border_radius="6px",
                background_color="#f8fafc",
            ),
        )

        output_plot = widgets.Output()
        output_plot.layout = widgets.Layout(width="100%", padding="0px 10px")

        header_widget = widgets.HTML(
            "<h3 style='text-align: center; color: #1e3a8a; margin: 0;'>Інтерактивна панель аналізу даних</h3>"
        )
        footer_widget = widgets.HTML(
            "<p style='text-align: center; color: #94a3b8; font-size: 11px; margin: 5px 0 0 0;'>Розроблено в Jupyter за допомогою ipywidgets</p>"
        )

        def update_dashboard(*args):
            col = dropdown_column.value
            start_idx, end_idx = slider_range.value

            sliced_data = df[col].iloc[start_idx : end_idx + 1].dropna()

            if not sliced_data.empty:
                lbl_count.value = f"{len(sliced_data)}"
                lbl_min.value = f"{sliced_data.min():.4f}"
                lbl_max.value = f"{sliced_data.max():.4f}"
                lbl_mean.value = f"{sliced_data.mean():.4f}"
                lbl_median.value = f"{sliced_data.median():.4f}"
            else:
                for lbl in [lbl_count, lbl_min, lbl_max, lbl_mean, lbl_median]:
                    lbl.value = "Немає даних"

            with output_plot:
                clear_output(wait=True)
                y_vals = sliced_data.values
                x_vals = sliced_data.index.values

                if len(y_vals) < 2:
                    print("Необхідно мінімум 2 точки у діапазоні для лінії.")
                    return

                fig, ax = plt.subplots(figsize=(6.5, 3.8))
                if len(y_vals) >= 4:
                    x_smooth = np.linspace(x_vals.min(), x_vals.max(), 300)
                    spline = make_interp_spline(x_vals, y_vals, k=3)
                    y_smooth = spline(x_smooth)
                    ax.plot(
                        x_smooth,
                        y_smooth,
                        color=picker_color.value,
                        linewidth=slider_width.value,
                        label=col,
                    )
                else:
                    ax.plot(
                        x_vals,
                        y_vals,
                        color=picker_color.value,
                        linewidth=slider_width.value,
                        marker="o",
                        label=col,
                    )

                ax.set_title(f"Плавний лінійкий графік параметру: {col}", fontsize=11)
                ax.set_xlabel("Індекс рядка", fontsize=9)
                ax.set_ylabel(col, fontsize=9)
                ax.legend(fontsize=8)
                plt.tight_layout()
                plt.show()
        dropdown_column.observe(update_dashboard, names="value")
        slider_width.observe(update_dashboard, names="value")
        picker_color.observe(update_dashboard, names="value")
        slider_range.observe(update_dashboard, names="value")
        gui_app = AppLayout(
            header=header_widget,
            left_sidebar=left_sidebar_panel,
            center=output_plot,
            right_sidebar=right_sidebar_panel,
            footer=footer_widget,
            pane_widths=["28%", "44%", "28%"],
            pane_heights=["40px", "360px", "25px"],
            grid_gap="15px",
        )
        gui_app.layout = widgets.Layout(
            border="1px solid #cbd5e1",
            padding="15px",
            border_radius="8px",
            background_color="#ffffff",
        )

        update_dashboard()
        display(gui_app)

Помилка: DataFrame порожній! Спочатку завантажте дані у Першій клітинці.
